In [ ]:
import sys
print(sys.executable)

import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

from torch.utils.data import DataLoader
import numpy as np
import sys, time, os

In [ ]:
###
### Part 1. Tokenization.
###

class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""

    def __init__(self, SOMETHING):
        # TODO: store all values you need in order to implement __call__ below.
        self.pad_token_id = ...     # Compulsory attribute.
        self.model_max_length = ... # Needed for truncation.

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.
           
           Args:
             texts:           The texts to tokenize.
             truncation:      Whether the texts should be truncated to model_max_length.
             padding:         Whether the tokenized texts should be padded on the right side.
             return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

           Returns:
             A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        
        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # TODO: Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.
        return BatchEncoding({'input_ids': ...})

    def __len__(self):
        """Return the size of the vocabulary."""
        return ...
    
    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)


In [ ]:
def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length.
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """
    # TODO: build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    special_tokens = [pad_token, unk_token, bos_token, eos_token]
    counter = Counter()
    with open(train_file, encoding='utf-8') as file:
        for line in file:
            text = line.strip()
            if text: counter.update(tokenize_fun(text))
    # print(counter)

    str_to_int = {}
    for token in special_tokens:
        str_to_int[token] = len(str_to_int)

    if max_voc_size is None:
        num_regular_tokens = None
    else:
        num_regular_tokens = max_voc_size - len(special_tokens)
        if num_regular_tokens < 0:
            raise ValueError("max_voc_size must be at least the number of special tokens")
    
    for token, _ in counter.most_common(num_regular_tokens):
          if token not in str_to_int:
              str_to_int[token] = len(str_to_int)

    # print(str_to_int)
    if max_voc_size: assert len(str_to_int) <= max_voc_size, "max_voc_size exeeded"

    int_to_str = {i: token for token, i in str_to_int.items()}

    # return A1Tokenizer(
    #     str_to_int=str_to_int,
    #     int_to_str=int_to_str,
    #     tokenize_fun=tokenize_fun,
    #     model_max_length=model_max_length,
    #     pad_token=pad_token,
    #     unk_token=unk_token,
    #     bos_token=bos_token,
    #     eos_token=eos_token,
    # )






print(lowercase_tokenizer("Let's test!!"))
build_tokenizer(train_file="./train.txt", max_voc_size=10, model_max_length=100)

['let', "'s", 'test', '!', '!']
{'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3, 'the': 4, ',': 5, '.': 6, 'of': 7, 'and': 8, 'in': 9}
